In [ ]:
import json
import os
from pathlib import Path
import zipfile

import fiona
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import pycountry
from pyquadkey2 import quadkey
import rasterio
from rasterio.transform import xy
from shapely.geometry import Point

In [ ]:
# Folder with WIA in Sharepoint
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_out_folder = base_dir + "data_out/"

# out folder for maps (hazards)
maps_hazards_folder = base_dir + "data_out/maps/hazards/"

# shapes subfolder
data_in_shapes = base_dir + "data_in/shapes/"

# meta rwi subfolder
data_in_rwi = base_dir + "data_in/rwi_meta/"
rwi_countries_zip = "relative-wealth-index-april-2021.zip"

# DBFS root
dbfs_base_dir = "./"
# WorldPop folder
wpop_dir = "WorldPop_rasters"
# indiv country rasters folder (constrained - total population)
country_pop_dir = "G2_CN_POP_R25A_100m"
# list of total_pop rasters available in DBFS
total_pop_rasters = [
    f.name
    for f in Path(
        data_in_folder + wpop_dir + "/" + country_pop_dir
    ).iterdir()
    if f.is_file()
]


In [ ]:
# define country/admin level/year
country = [
    # "Kenya",
    # "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]
country = country[0]
admin_level = "2"
year = "2025"

c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3 if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)


In [ ]:
# identify raster id for the country
# NOTE: assumes raster already downloaded in DBFS
country_raster_id = [
    data_in_folder + wpop_dir + "/" + country_pop_dir + "/" + next(
        (
            f for f in total_pop_rasters
            if (c_iso3.lower() in f) and (f"_{year}_" in f)
        ),
        "None"
    )
]

# geojson output file name
geo_filename = f"geojson_{c_iso3}_adm{admin_level}"
# Get geojson in shapes
geojson_file = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# read country shape geojson into gdf
gdf = gpd.read_file(
    data_in_shapes + f"{geojson_file[0]}"
).to_crs('EPSG:4326')

# column pcode and name (assume harmonized across CODs)
pcode_col = f"adm{admin_level}_pcode"
name_col = f"adm{admin_level}_name"


In [ ]:
# rwi meta zip file (2.4 km resolution)
# coverage: 135 LMIC countries (referenced in PNAS paper), not available in HDX
# if country in conflict, Meta shared country individually, e.g. Lebanon, Afghanistan, Ukraine, Myanmar, etc.
# make sure complying with UNICEF and Meta agreement for conflict countries data sharing
with zipfile.ZipFile(
    data_in_rwi + rwi_countries_zip,
    'r',
) as zip_file:
    # Inspect list of csv files and select the country for analysis
    rwi_country_csv = [f for f in zip_file.namelist() if f"{c_iso3}_" in f]

    # read selected rwi dataset into DataFrame
    with zip_file.open(rwi_country_csv[0]) as f:
        rwi_df = pd.read_csv(f, encoding="utf-8")
        # print shape as later tiles on edges will be dropped
        print(rwi_df.shape)
        # if "quadkey" column present, drop it (computed later)
        if "quadkey" in rwi_df.columns:
            rwi_df.drop(columns="quadkey", inplace=True)
    
# Build RWI GeoDataFrame (points)
rwi_gdf = gpd.GeoDataFrame(
    rwi_df,
    geometry=gpd.points_from_xy(
        rwi_df["longitude"], rwi_df["latitude"]
    ),
    crs="EPSG:4326"
)

# Delete the Pandas DataFrame to free up memory
del rwi_df


In [ ]:
# create population dataframe from raster
with rasterio.open(
    # Open the dataset
    country_raster_id[0]
) as dataset:
    # Assumes exists one band only with population, print basic info
    print("Total number of bands:", dataset.count)
    # Assumes CRS is EPSG:4326
    print("Raster CRS:", dataset.crs)
    print("Raster no-data:", dataset.nodata)
    
    # Data: WorldPop rasters
    wpop_raster = dataset.read(1)
    print("Band shape:", wpop_raster.shape)

    # Get row, col indices of non-nodata pixels
    rows, cols = np.where(wpop_raster != dataset.nodata)

    # Get coordinates of valid pixels
    lons, lats = xy(dataset.transform, rows, cols)

    # Build geoDataFrame (consider Koalas if memory issues arise later)
    pop_gdf = gpd.GeoDataFrame(
        {
            'geometry': gpd.points_from_xy(lons, lats),
            f"tot_pop_{year}": wpop_raster[rows, cols]
        },
        crs="EPSG:4326",
    )

# pop. shape with data
print(pop_gdf.shape)
# pop. country total sum check
print(pop_gdf[f"tot_pop_{year}"].sum())


In [ ]:
# Spatial join: point within polygon (excludes boundary)
rwi_j_df = gpd.sjoin(
    rwi_gdf,
    gdf[[pcode_col, name_col, "geometry"]],
    how="left",
    predicate="within",
).drop(
    # back to df: geometry no longer needed
    columns=["index_right", "geometry"]
)

# Check if tiles assigned to more than one pcode
# NOTE: within should not produce duplicates, just in case check
if len(rwi_j_df) != len(rwi_gdf):
    raise Exception("Stop execution: Rwi tiles assigned to more than one pcode")
else:
    # Delete the GeoDataFrame to free up memory
    del rwi_gdf
    # print number of tiles on edges
    print(
        f"Tiles with unassigned pcodes: {rwi_j_df[pcode_col].isna().sum()}"
    )
    # remove the tiles on edges
    rwi_j_df.dropna(subset=[pcode_col], inplace=True)
    # print shape after dropping edge tiles
    print(rwi_j_df.shape)


In [ ]:
# Spatial join for population (excludes boundary)
# NOTE: pcodes could be assigned after population aggregation at quadkeys
# (as selected zoom is higher than pop resolution)
pop_j_gdf = gpd.sjoin(
    pop_gdf,
    gdf[[pcode_col, "geometry"]],
    how="left",
    predicate="within",
).drop(
    columns=["index_right"]
)

# Check if tiles assigned to more than one pcode
# NOTE: within should not produce duplicates, just in case check
if len(pop_j_gdf) != len(pop_gdf):
    raise Exception("Stop execution: Pop tiles assigned to more than one pcode")
else:
    # Delete the GeoDataFrame to free up memory
    del pop_gdf
    # print number of tiles on edges
    print(
        f"Tiles with unassigned pcodes: {pop_j_gdf[pcode_col].isna().sum()}"
    )
    # remove the tiles on edges
    pop_j_gdf.dropna(subset=[pcode_col], inplace=True)
    # print shape after dropping edge tiles
    print(pop_j_gdf.shape)


##### Determine the bing tile quadkey (at zoom level 14) for each RWI tile (2.4km^2)
- Note zoom level 14 is not arbitrary, it is the lowest zoom that provides a unique identifier for 2.4km^2 tiles
- Also apply same zoom level 14 for each point in the population gdf
- Sum the population per level 14 tile (knowing population tiles are smaller: 100m grid tiles)

In [ ]:
# create 'quadkey' column in rwi_j_df (zoom level 14)
# NOTE: revise workflow if not performant
zoom_level = 14

rwi_j_df['quadkey'] = rwi_j_df.apply(
    lambda x: str(
        quadkey.from_geo((x['latitude'], x['longitude']), zoom_level)
    ),
    axis=1,
)

# create 'quadkey' column in pop_j_gdf (zoom level 14)
pop_j_gdf['quadkey'] = [
    str(quadkey.from_geo((lat, lon), zoom_level))
    for lat, lon in zip(
        pop_j_gdf.geometry.y.values, pop_j_gdf.geometry.x.values
    )
]

# Check tiles-to-quadkey relations
if len(rwi_j_df.quadkey.unique()) != len(rwi_j_df):
    raise Exception("Stop execution: RWI-to-quadkeys should be 1:1")
else:
    # Check pop-to-quadkey 1:m relation
    print(
        f"Number of unique pop quadkeys: {len(pop_j_gdf.quadkey.unique())}"
    )
    # back to df: geometry no longer needed
    pop_j_df = pop_j_gdf.drop(columns=["geometry"])
    # pop. country total sum check
    # NOTE: should be less given edge tiles dropped
    print(pop_j_df[f"tot_pop_{year}"].sum())
    # Delete the GeoDataFrame to free up memory
    del pop_j_gdf


##### Calculate population weighted RWI by admin area
- Join RWI with Population on Quadkeys (analysis if only data in both)
    - This approach does not wash out values
    - Considers RWI is representative
        - Note RWI tiles present in admin area may represent only a small pop fraction
- Calculate a population-derived weight for each zoom level 14 RWI tile
- Use the weight value to calculate a weighted RWI value and aggregate to the administrative unit level

In [ ]:
# inner join rwi_j_df and pop_j_df on quadkey/geo_id
rwi_j_df = rwi_j_df.merge(
    pop_j_df,
    on=[pcode_col, 'quadkey'],
    how='inner',
    validate='1:m',
)

# Check rwi with pop to-quadkey 1:m relation
print(
    f"Number of unique quadkeys with rwi and pop: {len(rwi_j_df.quadkey.unique())}"
)
# pop. country total sum check
# NOTE: should be less given missing RWI tiles
print(rwi_j_df[f"tot_pop_{year}"].sum())

# Delete the pop DataFrame to free up memory
del pop_j_df


In [ ]:
# sum population by quadkey zoom level 14 (where available RWI tiles)
# join population per admin area polygon
rwi_j_df = rwi_j_df.groupby(
    'quadkey', as_index=False
).agg(
    {
        f"tot_pop_{year}": 'sum',
        pcode_col: 'first',
    }
).rename(
    columns={f"tot_pop_{year}": f"q_pop_{year}"}
).merge(
    rwi_j_df.groupby(
        pcode_col, as_index=False
    ).agg({f"tot_pop_{year}": 'sum'}).rename(
        columns={f"tot_pop_{year}": f"geo_pop_{year}"}
    ),
    on=pcode_col,
    how="left",
    validate='m:1',
).merge(
    rwi_j_df.drop_duplicates(
        subset=[pcode_col, 'quadkey']
    )[
        [pcode_col, name_col, 'quadkey', 'rwi']
    ],
    on=[pcode_col, 'quadkey'],
    how='left',
    validate='1:1',
)

# pop weight: pop in quadkey (where RWI tiles) by pop in admin area (where quadkey "within")
rwi_j_df['pop_weight'] = (
    rwi_j_df[f"q_pop_{year}"] / rwi_j_df[f"geo_pop_{year}"]
)


In [ ]:
# pop-weighted RWI (RWI score by 'pop_weight', where rwi and pop available)
rwi_j_df['rwi_weighted'] = rwi_j_df['rwi'] * rwi_j_df['pop_weight']

# RWI at the admin unit level, groupby by pcode and sum pop-weighted RWI
# merge with gdf for missing data in pcodes (if any)
geo_rwi_df = rwi_j_df.groupby(
    [pcode_col, name_col], as_index=False
).agg({'rwi_weighted': 'sum'}).merge(
    gdf[[pcode_col, name_col]],
    on=[pcode_col, name_col],
    how='right',
    validate='1:1',
).sort_values(by=pcode_col).rename(
    columns={'rwi_weighted': f"rwi_weighted_pop_{year}"}
)


In [ ]:
# 0/1 binary mapping for wia: below or above mean/median
rwi_mean = geo_rwi_df[f"rwi_weighted_pop_{year}"].mean()
rwi_med = geo_rwi_df[f"rwi_weighted_pop_{year}"].median()

# series mapping into new columns
geo_rwi_df["rwi_below_mean"] = (geo_rwi_df[f"rwi_weighted_pop_{year}"] <= rwi_mean).astype(int)
geo_rwi_df["rwi_below_median"] = (geo_rwi_df[f"rwi_weighted_pop_{year}"] <= rwi_med).astype(int)

# ensure NaN in binary mapping if NaN in rwi_weighted
geo_rwi_df.loc[
    geo_rwi_df[f"rwi_weighted_pop_{year}"].isnull(),
    ["rwi_below_mean", "rwi_below_median"],
] = np.nan


In [ ]:
# save RWI data (fill no-data for missing RWI/pop, if any)
geo_rwi_df.fillna("no-data").to_excel(
    maps_hazards_folder + f"{c_iso3}_adm{admin_level}_{year}_RWI.xlsx",
    index=False,
    sheet_name=f"{c_iso3}_adm{admin_level}_{year}",
)


In [ ]:
# customed px continous color scale: name
color_scale_name = "RdYlGn"
color_names = eval(f"px.colors.diverging.{color_scale_name}")

# customed color scale with color for NaNs
color_nan = "Gray"
customed_color_nan = [
    [0, color_nan],
    [0.001, color_nan],
    [0.001, color_names[0]],
]
customed_color_rem = [
    [(i + 1) / (len(color_names) - 1), a_color]
    for i, a_color in enumerate(color_names[1:])
]
customed_color_scale = customed_color_nan + customed_color_rem

# COMMAND ----------

# set the range of RWI before adding the NA values (-999)
full_range = [
    geo_rwi_df[f"rwi_weighted_pop_{year}"].min() - 0.5,
    geo_rwi_df[f"rwi_weighted_pop_{year}"].max(),
]


In [ ]:
# Disable automatic rendering
pio.renderers.default = None

# plotly express choropleth map of country RWI by admin area polygon
rwi_fig = px.choropleth(
    geo_rwi_df.fillna(-999),
    geojson=json.loads(gdf.to_json()),
    featureidkey=f"properties.{pcode_col}",
    locations=pcode_col,
    color=f"rwi_weighted_pop_{year}",
    color_continuous_scale=customed_color_scale,
    range_color=full_range,
    projection="mercator",
    width=1200,
    height=900,
)
rwi_fig.update_geos(fitbounds="locations", visible=False)
rwi_fig.update_layout(
    title=dict(
        text=f"{c_iso3}_adm{admin_level} RWI (weighted by {year} population)",
        x=0.5,  # x position (0-left, 1-right)
        y=0.97,  # y position (0-bottom, 1-top in normalized coordinates)
    ),
    margin={"r": 0, "t": 0, "l": 0, "b": 0}
)
rwi_fig.update_coloraxes(colorbar_len=0.65)
rwi_fig.update_traces(marker_line_color="Gainsboro", marker_line_width=0.5)


In [ ]:
# write figure as png direct locally
rwi_fig.write_image(
    maps_hazards_folder + f"{c_iso3}_adm{admin_level}_{year}_RWI.png",
    format="png",
    scale=2,
)
